PROJET_DIT


Chargement des packages

In [1]:

# Importation des bibliothèques nécessaires
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import classification_report
import joblib # Pour stocker le modèle
from sklearn.ensemble import RandomForestClassifier

Base de données insecurite alimentaire

Importation et nettoyage des données

In [ ]:



data = pd.read_excel("D:/PROJET_DIT-20250506T153458Z-001/MES_PROJETS/DIEMMME/DDDIEMMME.xlsx", sheet_name="Feuil2")

# Aperçu des données
print("Aperçu des données :")
#print(data.head())


In [ ]:
import numpy as np
# 1. Vérifier les valeurs manquantes
print("Valeurs manquantes avant nettoyage:")
print(data.isnull().sum())

In [ ]:
# Remplacer les valeurs manquantes par la médiane
# pour les colonnes numériques uniquement
numeric_cols = data.select_dtypes(include=np.number).columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())


In [ ]:
# 2. Suppression des doublons
print(f"Nombre de doublons avant suppression: {data.duplicated().sum()}")
data.drop_duplicates(inplace=True)

In [ ]:
# 3. Traitement des valeurs aberrantes
# Remplacer les valeurs aberrantes par la médiane ou les supprimer
import numpy as np

for colonne in data.select_dtypes(include=np.number).columns:
    seuil_min, seuil_max = data[colonne].quantile(0.05), data[colonne].quantile(0.95)
    data[colonne] = np.where((data[colonne] < seuil_min) | (data[colonne] > seuil_max),
                             data[colonne].median(),
                             data[colonne])


In [ ]:
data.shape

NORMALISATION DES VARIABLE NUMERIQUE

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
for col in data.select_dtypes(include=['object']).columns:  # Sélectionner les colonnes non numériques
    # Convert the column to string type before applying LabelEncoder
    data[col] = data[col].astype(str)
    data[col] = label_encoder.fit_transform(data[col])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Appliquer OneHotEncoder
# Replace 'sparse' with 'sparse_output'
from sklearn.preprocessing import OneHotEncoder

# Appliquer OneHotEncoder
# Replace 'sparse' with 'sparse_output'
one_hot_encoder = OneHotEncoder(sparse_output=False)
data_encoded = one_hot_encoder.fit_transform(data.select_dtypes(include=['object']))
data_encoded = pd.DataFrame(data_encoded, columns=one_hot_encoder.get_feature_names_out(data.select_dtypes(include=['object']).columns))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data[data.select_dtypes(include=['float', 'int']).columns] = scaler.fit_transform(data.select_dtypes(include=['float', 'int']))


In [ ]:
# Enregistrer le dataset encodé
chemin_projet = "D:/PROJET_DIT-20250506T153458Z-001/MES_PROJETS/"  # Remplacez par le chemin de votre répertoire
data.to_csv(chemin_projet + "data_encoded_1.csv", index=False)

print("Le fichier a été enregistré avec succès dans le répertoire de votre projet.")



In [ ]:
import pandas as pd

# Charger le fichier encodé
chemin_projet = "D:/PROJET_DIT-20250506T153458Z-001/MES_PROJETS/"
data_encoded = pd.read_csv(chemin_projet + "data_encoded_1.csv")

# Vérification rapide
print("Aperçu du fichier encodé :")
#print(data_encoded.head())
#print("Colonnes disponibles :", data_encoded.columns)

In [ ]:
data_encoded.shape

In [ ]:
import pandas as pd

# Étape 1 : Définir la variable cible 'insécurité_alimentaire'
def definir_insécurité_alimentaire(row):
    # Insécurité sévère : au moins 1 privation critique avec score > 1.5
    if sum(row[col] > 1.5 for col in [
        'q605_1_ne_plus_avoir_de_nourriture_pas_suffisamment_d_argent',
        'q606_1_avoir_faim_mais_ne_pas_manger',
        'q607_1_passer_toute_une_journee_sans_manger'
    ]) >= 1:
        return 2
    # Insécurité modérée : au moins 2 privations légères/modérées avec score entre 0 et 1.5
    elif sum(0 < row[col] <= 1.5 for col in [
        'q600_inquiets_de_ne_pas_avoir_suffisamment_de_nourriture',
        'q601_ne_pas_manger_nourriture_saine_nutritive',
        'q602_manger_nourriture_peu_variee',
        'q603_sauter_un_repas',
        'q604_manger_moins_que_ce_que_vous_auriez_du'
    ]) >= 2:
        return 1
    # Aucune insécurité
    else:
        return 0



 PREPARATION DES DONNEES POUR LE MODELE

In [ ]:
# Étape 2 : Appliquer la fonction au DataFrame encodé
data['insécurité_alimentaire'] = data_encoded.apply(definir_insécurité_alimentaire, axis=1)

# Étape 3 : Filtrer les cas modérés ou sévères
data_filtrée = data[data['insécurité_alimentaire'].isin([1, 2])]
data_encoded_filtré = data_encoded.loc[data_filtrée.index]  # Garder les mêmes lignes dans le dataset encodé

# Étape 4 : Calculer les corrélations avec la variable cible
data_encoded_filtré['insécurité_alimentaire'] = data_filtrée['insécurité_alimentaire']  # Ajouter la cible
correlations = data_encoded_filtré.corr()

# Étape 5 : Sélectionner les 10 variables les plus corrélées
top_features = correlations['insécurité_alimentaire'].abs().sort_values(ascending=False).iloc[1:11]
print("🔍 Les 10 variables les plus corrélées :")
print(top_features)

# Étape 6 : Créer un nouveau dataset avec ces variables + la cible
top_features_names = top_features.index.tolist()
data_reduced = data_encoded_filtré[top_features_names + ['insécurité_alimentaire']]
print("📊 Nouveau dataset réduit :")
#print(data_reduced.head())


In [ ]:
# Variables explicatives (features) et cible (target)
X = data_reduced.drop(columns=['insécurité_alimentaire'])  # Toutes les colonnes sauf la cible
y = data_reduced['insécurité_alimentaire']  # La variable cible # C

DIVISION EN TRAIN TEST DU DATASET

In [ ]:
from sklearn.model_selection import train_test_split

# Division 80% entraînement, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vérifier les tailles des ensembles
print("Taille de l'ensemble d'entraînement :", X_train.shape)
print("Taille de l'ensemble de test :", X_test.shape)


In [ ]:
# Remapping des classes : 1 → 0 (modérée), 2 → 1 (sévère)
y_train_mapped = y_train.replace({1: 0, 2: 1})
y_test_mapped = y_test.replace({1: 0, 2: 1})


ENTRAINEMENT 1 DU MODELE

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

# 🔁 Modèle de base
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train_mapped)

# 🔍 Étape 1 : extraire les importances
importances = rf.feature_importances_
feature_names = X_train.columns

# 🔍 Étape 2 : sélectionner les 5 variables les plus importantes
indices_top5 = np.argsort(importances)[-5:]  # indices des 5 plus importantes
selected_features = feature_names[indices_top5]

print("✅ 5 variables sélectionnées :", selected_features.tolist())

# ✅ Étape 3 : reconstruire les DataFrames
X_train_selected_df = X_train[selected_features]
X_test_selected_df = X_test[selected_features]

# 🔁 Réentraînement avec les 5 variables
rf_selected = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selected.fit(X_train_selected_df, y_train_mapped)
preds = rf_selected.predict(X_test_selected_df)

# 📊 Évaluation
print("📊 Rapport après sélection des 5 features :")
print(classification_report(y_test_mapped, preds))

# 🔍 Validation croisée
cv_scores = cross_val_score(rf_selected, X_train_selected_df, y_train_mapped, cv=5, scoring='accuracy')
print(f"✅ Score moyen CV (Random Forest 5 features) : {cv_scores.mean():.4f}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Prédictions des probabilités
y_train_proba = rf_selected.predict_proba(X_train_selected_df)[:, 1]
y_test_proba = rf_selected.predict_proba(X_test_selected_df)[:, 1]

# Courbes ROC
fpr_train, tpr_train, _ = roc_curve(y_train_mapped, y_train_proba)
fpr_test, tpr_test, _ = roc_curve(y_test_mapped, y_test_proba)

# AUC
auc_train = auc(fpr_train, tpr_train)
auc_test = auc(fpr_test, tpr_test)

# 🎨 Affichage
plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, label=f"Ensemble d'entraînement (AUC = {auc_train:.2f})", color='blue')
plt.plot(fpr_test, tpr_test, label=f"Ensemble de test (AUC = {auc_test:.2f})", color='orange')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Classifieur aléatoire')

plt.title("Courbe ROC", fontsize=14)
plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


ENTRAINEMENT 2 DU MODELE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn.model_selection import cross_val_score

# 🔹 Modèle initial pour la sélection de features
rf_base = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)
rf_base.fit(X_train, y_train_mapped)

# 🔹 Sélection des features importantes
selector = SelectFromModel(rf_base, threshold=None, prefit=True)
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)


# 🔍 Récupération des noms des features sélectionnées
selected_mask = selector.get_support()
feature_names_rf = X_train.columns[selected_mask].tolist()

# 🧪 Reconstruction de X_test avec les noms des features
X_test_rf = X_test[feature_names_rf].values

# 🔹 Modèle 1 : Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_selected, y_train_mapped)
rf_preds = rf_model.predict(X_test_selected)
rf_proba = rf_model.predict_proba(X_test_selected)[:, 1]

# 🔹 Modèle 2 : XGBoost
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train_selected, y_train_mapped)
xgb_preds = xgb_model.predict(X_test_selected)
xgb_proba = xgb_model.predict_proba(X_test_selected)[:, 1]

# 🔹 Courbes ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test_mapped, rf_proba)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test_mapped, xgb_proba)
auc_rf = auc(fpr_rf, tpr_rf)
auc_xgb = auc(fpr_xgb, tpr_xgb)

# 🔹 Affichage ROC comparatif
plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.2f})", color='blue')
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {auc_xgb:.2f})", color='orange')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Classifieur aléatoire')

plt.title("Courbe ROC - Comparaison des modèles", fontsize=14)
plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

# 🔹 Évaluation textuelle
print("📊 Rapport Random Forest :")
print(classification_report(y_test_mapped, rf_preds))

print("📊 Rapport XGBoost :")
print(classification_report(y_test_mapped, xgb_preds))

# 🔹 Validation croisée
rf_cv = cross_val_score(rf_model, X_train_selected, y_train_mapped, cv=5, scoring='roc_auc')
xgb_cv = cross_val_score(xgb_model, X_train_selected, y_train_mapped, cv=5, scoring='roc_auc')

print(f"✅ Score moyen AUC CV - Random Forest : {rf_cv.mean():.4f}")
print(f"✅ Score moyen AUC CV - XGBoost       : {xgb_cv.mean():.4f}")


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score
import pandas as pd

# 🔹 Prédictions sur l'ensemble d'entraînement
rf_train_pred = rf_model.predict(X_train_selected)
xgb_train_pred = xgb_model.predict(X_train_selected)

# 🔹 Calcul des métriques pour Random Forest
rf_metrics = {
    'Accuracy': [accuracy_score(y_train_mapped, rf_train_pred), accuracy_score(y_test_mapped, rf_preds)],
    'AUC': [roc_auc_score(y_train_mapped, rf_train_pred), roc_auc_score(y_test_mapped, rf_preds)],
    'Recall': [recall_score(y_train_mapped, rf_train_pred), recall_score(y_test_mapped, rf_preds)]
}

# 🔹 Calcul des métriques pour XGBoost
xgb_metrics = {
    'Accuracy': [accuracy_score(y_train_mapped, xgb_train_pred), accuracy_score(y_test_mapped, xgb_preds)],
    'AUC': [roc_auc_score(y_train_mapped, xgb_train_pred), roc_auc_score(y_test_mapped, xgb_preds)],
    'Recall': [recall_score(y_train_mapped, xgb_train_pred), recall_score(y_test_mapped, xgb_preds)]
}

# 🔹 Création des tableaux
# 🔹 Tableau Random Forest
rf_table = pd.DataFrame({
    'Métrique': ['Accuracy', 'AUC', 'Recall'],
    'Train': [
        accuracy_score(y_train_mapped, rf_train_pred),
        roc_auc_score(y_train_mapped, rf_train_pred),
        recall_score(y_train_mapped, rf_train_pred)
    ],
    'Test': [
        accuracy_score(y_test_mapped, rf_preds),
        roc_auc_score(y_test_mapped, rf_preds),
        recall_score(y_test_mapped, rf_preds)
    ]
})

# 🔹 Tableau XGBoost
xgb_table = pd.DataFrame({
    'Métrique': ['Accuracy', 'AUC', 'Recall'],
    'Train': [
        accuracy_score(y_train_mapped, xgb_train_pred),
        roc_auc_score(y_train_mapped, xgb_train_pred),
        recall_score(y_train_mapped, xgb_train_pred)
    ],
    'Test': [
        accuracy_score(y_test_mapped, xgb_preds),
        roc_auc_score(y_test_mapped, xgb_preds),
        recall_score(y_test_mapped, xgb_preds)
    ]
})

# 🔹 Affichage
print("\n📋 Performance - Random Forest")
print(rf_table)

print("\n📋 Performance - XGBoost")
print(xgb_table)



In [ ]:
X_test_rf = X_test[feature_names_rf]
rf_preds = rf_model.predict(X_test_rf)
print(rf_preds)

Prédiction sur les bases train et test

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

# Prédire les classes sur les ensembles d'entraînement et de test
y_train_pred = rf_model.predict(X_train_selected)
y_test_pred = rf_model.predict(X_test_selected)

Performances du modèle

In [ ]:
# Calculer les mesures de performance
train_accuracy = accuracy_score(y_train_mapped, y_train_pred)
test_accuracy = accuracy_score(y_test_mapped, y_test_pred)
train_auc = roc_auc_score(y_train_mapped, y_train_pred)
test_auc = roc_auc_score(y_test_mapped, y_test_pred)
train_recall = recall_score(y_train_mapped, y_train_pred)
test_recall = recall_score(y_test_mapped, y_test_pred)

# Créer le tableau d'évaluation des performances
performance_table = pd.DataFrame({
    'Métrique': ['Accuracy', 'AUC', 'Recall'],
    'Ensemble d\'entraînement': [train_accuracy, train_auc, train_recall],
    'Ensemble de test': [test_accuracy, test_auc, test_recall]
})

# Afficher le tableau d'évaluation des performances
print(performance_table)

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model, X_train_selected, y_train_mapped, cv=5, scoring='f1')
print(f"F1 scores par fold : {scores}")
print(f"F1 moyen : {scores.mean():.4f} ± {scores.std():.4f}")


DEPLOIMENT DU MODELE

In [ ]:
import joblib
joblib.dump(rf_model, "modele_food_insecurity.pkl")
joblib.dump(xgb_model, "modele_food_insecurity_xgb1.pkl")
# Enregistrement du modèle avec joblib

In [ ]:
# 🔹 Sauvegarde des modèles
joblib.dump(rf_model, "modele_food_insecurity.pkl")
joblib.dump(xgb_model, "modele_food_insecurity_xgb1.pkl")

print("🎉 Modèles sauvegardés avec succès !")